# Trivariate Gaussian process emulation of joint contact forces

The first notebook of this project, `jcf-gp-colab.ipynb`, builds a univariate Gaussian process (GP) emulator of the joint contact forces computed by OpenSim, one joint at a time. This notebook models the hip, the knee and the ankle jointly with a trivariate Matérn model over the stride inputs. The output of each joint is one scalar per stride, the peak joint contact force over the gait cycle. Each joint keeps its own variance, input range and nugget effect, and every pair of joints has a cross-correlation, because the contact forces of one stride at the three joints are produced by the same ground reaction force and by muscles that cross more than one joint.

With joint-specific ranges the covariance matrix has no Kronecker structure, and it is treated as a dense matrix with $N = 3 \times 480 = 1{,}440$ rows. Maximum likelihood estimation requires the Cholesky factorization of this matrix at every evaluation of the likelihood. The notebook fits the model with the adaptive block mixed-precision Cholesky factorization of Salvaña (2026), which stores each block of the Cholesky factor in the lowest precision that a guaranteed error budget allows.

The notebook has two parts. It loads the data, and it fits the trivariate model by maximum likelihood with the mixed-precision factorization. It runs on any GPU runtime. In Colab, choose Runtime, then Change runtime type, then a GPU.

## Setup

In [ ]:
import json, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import minimize

assert torch.cuda.is_available(), "No GPU is attached. Choose Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{GPU_NAME}, {GPU_GB:.1f} GB")

## Data

The data are those of the first notebook. One row is one stride of the right leg, and there are 480 training strides from twelve participants. The inputs are three ground reaction force curves, five joint angle curves, the belt speed, the body mass and the height. The outputs are the hip, knee and ankle joint contact force curves computed by OpenSim, in body weights (BW), time normalized to 101 points of the gait cycle. This notebook reduces each output curve to its peak over the gait cycle, so the output of a stride is one peak joint contact force per joint. The variable `TARGET` below selects the peak, the mean over the gait cycle, or a single gait point.

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1CI5OAxzB9qYI9vIxW1f0cGE4SN1hhyBj" -q

In [ ]:
x_train = pd.read_csv("data/x-train.csv")
y_train = pd.read_csv("data/y-train.csv")
x_train.shape, y_train.shape

Each stride becomes the vector $\mathbf{x}_i$ of its eight input curves and three scalars, standardized on the training strides exactly as in the first notebook.

In [ ]:
GRID = np.arange(101)
JOINTS = ["hip", "knee", "ankle"]
INPUT_CURVES = ["grf_vertical_BW", "grf_anteroposterior_BW", "grf_mediolateral_BW",
                "hip_flexion_deg", "hip_adduction_deg", "hip_rotation_deg", "knee_flexion_deg", "ankle_angle_deg"]
TARGET = "peak"                                            # "peak", "mean", or a gait point from 0 to 100

def curve(df, name):
    return df[[f"{name}_t{t:03d}" for t in GRID]].to_numpy()

def build_features(x, x_ref):
    cols = []
    for name in INPUT_CURVES:
        C, C_ref = curve(x, name), curve(x_ref, name)
        cols.extend(((C - C_ref.mean(axis=0)) / C_ref.std()).T)
    for name in ["belt_speed_mps", "mass_kg", "height_m"]:
        cols.append((x[name].to_numpy() - x_ref[name].mean()) / x_ref[name].std())
    return np.array(cols).T

F_train = build_features(x_train, x_train)
F_train.shape

The Gaussian process models the residual of each joint, the output minus the mean $m_j$ of the training strides. The residuals of joint $j$ are divided by the standard deviation $s_j$ of its training outputs.

In [ ]:
def target(df, joint):
    C = curve(df, f"{joint}_jcf_BW")                                              # (strides, 101 gait points), in BW
    if TARGET == "peak":
        return C.max(axis=1)
    if TARGET == "mean":
        return C.mean(axis=1)
    return C[:, int(TARGET)]

Y_train = np.stack([target(y_train, joint) for joint in JOINTS])                 # (3 joints, 480 strides), in BW
mean = Y_train.mean(axis=1)                                                       # mean of each joint
scale = Y_train.std(axis=1)                                                       # s_j, in BW
Z_train = (Y_train - mean[:, None]) / scale[:, None]                              # standardized residuals
Z_train.shape

## Trivariate Matérn model

Let $i = 1, \dots, n$ index the training strides, with $n = 480$, and let $j \in \{1, 2, 3\}$ index the hip, the knee and the ankle. The input of stride $i$ is the vector $\mathbf{x}_i \in \mathbb{R}^{811}$, and $y_{ij}$ is the output of joint $j$ at stride $i$. Each joint follows the decomposition
$$
y_{ij} = m_j + f_j(\mathbf{x}_i) + \varepsilon_{ij}, \qquad \varepsilon_{ij} \sim \mathcal{N}(0, \sigma_{\varepsilon, j}^2),
$$
where $m_j$ is the mean of joint $j$ and $\sigma_{\varepsilon, j}^2$ is the nugget effect of joint $j$, and $(f_1, f_2, f_3)$ is a trivariate Gaussian process with a covariance function for every pair of joints.

Write $h$ for the root mean squared difference between the standardized inputs of two strides. The covariance function is
$$
\operatorname{Cov}\!\big(f_j(\mathbf{x}), f_k(\mathbf{x}')\big) = \beta_{jk}\, \sigma_j \sigma_k \;\, \gamma_{jk}\, \mathcal{M}\!\Big(\frac{h}{\ell_{jk}}\Big), \qquad \mathcal{M}(d) = \Big(1 + \sqrt{5}\, d + \tfrac{5}{3} d^2\Big) e^{-\sqrt{5}\, d},
$$
where $\mathcal{M}$ is the Matérn correlation function with smoothness $5/2$. Joint $j$ has its own variance $\sigma_j^2$ and input range $\ell_j$, and the parameters $\beta_{jk}$ form a correlation matrix. For a pair of joints the range and the factor $\gamma$ follow from the ranges of the two joints,
$$
\frac{1}{\ell_{jk}^2} = \frac{1}{2} \Big( \frac{1}{\ell_j^2} + \frac{1}{\ell_k^2} \Big), \qquad \gamma_{jk} = \Big( \frac{2\, \ell_j \ell_k}{\ell_j^2 + \ell_k^2} \Big)^{5/2}.
$$
This construction is valid for every correlation matrix $(\beta_{jk})$ and in any input dimension (Gneiting, Kleiber and Schlather, 2010, and Apanasovich, Genton and Sun, 2012). For a single joint the covariance function reduces to $\sigma_j^2\, \mathcal{M}(h / \ell_j)$, the Matérn covariance function of the univariate emulator. The cross-correlation between the outputs of joints $j$ and $k$ at the same stride is $\rho_{jk} = \beta_{jk}\, \gamma_{jk}$.

The model has eleven parameters, and they are estimated by maximizing the likelihood of the stacked residuals $\mathbf{y}$, a vector of length $N = 1{,}440$ with covariance matrix $\boldsymbol{\Sigma}$. The variance of the hip is removed analytically. Write $\boldsymbol{\Sigma} = \sigma_1^2\, \bar{\boldsymbol{\Sigma}}$, where $\bar{\boldsymbol{\Sigma}}$ depends on the ranges, the parameters $\beta_{jk}$, the variance ratios $\sigma_j^2 / \sigma_1^2$ and the relative nugget effects $\eta_j = \sigma_{\varepsilon, j}^2 / \sigma_1^2$. For fixed values of these ten parameters the likelihood is maximized by $\hat{\sigma}_1^2 = \mathbf{y}^\top \bar{\boldsymbol{\Sigma}}^{-1} \mathbf{y} / N$, and the negative log likelihood is
$$
-\log p = \tfrac{1}{2} \Big( N \log \hat{\sigma}_1^2 + \log \det \bar{\boldsymbol{\Sigma}} + N + N \log 2\pi \Big).
$$
Each evaluation needs the log determinant of $\bar{\boldsymbol{\Sigma}}$ and one linear solve, and both come from its Cholesky factor.

The class below holds the model. It orders the points by joint, then stride, and it returns any block of $\bar{\boldsymbol{\Sigma}}$ on the GPU in double precision, so the full matrix is never stored.

In [ ]:
def matern(d):
    s = 5.0**0.5 * d
    return (1.0 + s + s * s / 3.0) * torch.exp(-s)

def correlation_matrix(raw):
    # 3 x 3 correlation matrix from three unconstrained numbers, through the angles of its Cholesky factor
    a = (np.pi / 2) * np.tanh(np.asarray(raw, dtype=float))
    L = np.array([[1.0, 0.0, 0.0],
                  [np.sin(a[0]), np.cos(a[0]), 0.0],
                  [np.sin(a[1]), np.cos(a[1]) * np.sin(a[2]), np.cos(a[1]) * np.cos(a[2])]])
    return L @ L.T

def pair_ranges(l):
    # 1 / l_jk and gamma_jk for every pair of joints
    a2 = 1.0 / l**2
    inverse = np.sqrt((a2[:, None] + a2[None, :]) / 2)
    return inverse, (np.sqrt(a2[:, None] * a2[None, :]) / inverse**2) ** 2.5

class TrivariateMatern:
    def __init__(self, F, Z, block_size=120):
        self.n, self.N = Z.shape[1], Z.size
        Ft = torch.tensor(F, dtype=torch.float64, device=DEVICE)
        self.H = torch.cdist(Ft, Ft, compute_mode="donot_use_mm_for_euclid_dist") / F.shape[1]**0.5      # h between strides
        self.y = torch.tensor(Z.reshape(-1), dtype=torch.float64, device=DEVICE)                          # stacked residuals
        index = torch.arange(self.N, device=DEVICE)
        self.joint, self.stride = index // self.n, index % self.n
        self.blocks = [slice(a, min(a + block_size, self.N)) for a in range(0, self.N, block_size)]

    def set_parameters(self, theta):
        # theta = log input ranges (3), correlation angles (3), log variance ratios of knee and ankle (2), log relative nugget effects (3)
        l = np.exp(theta[0:3])
        beta = correlation_matrix(theta[3:6])
        v = np.concatenate([[1.0], np.exp(theta[6:8])])
        eta = np.exp(theta[8:11])
        inverse_l, gamma = pair_ranges(l)
        rho = beta * gamma                                                         # cross-correlations at the same stride
        as_tensor = lambda a: torch.tensor(a, dtype=torch.float64, device=DEVICE)
        self.amplitude = as_tensor(rho * np.sqrt(np.outer(v, v)))
        self.inverse_l = as_tensor(inverse_l)
        self.nugget = as_tensor(eta)[self.joint]                                   # relative nugget effect of every point
        self.m = float(eta.min())                                                  # lower bound on the smallest eigenvalue
        self.parameters = dict(l=l, v=v, eta=eta, rho=rho)

    def block(self, rows, cols):
        # block of the covariance matrix between two sets of points, without the nugget effect
        pair = (self.joint[rows][:, None], self.joint[cols][None, :])
        h = self.H[self.stride[rows]][:, self.stride[cols]]
        return self.amplitude[pair] * matern(h * self.inverse_l[pair])

## Adaptive block mixed-precision Cholesky factorization

A matrix entry takes 8 bytes in double precision (`fp64`), 4 bytes in single precision (`fp32`), 2 bytes in `bfloat16` (`bf16`) and 1 byte in the 8-bit format `fp8`. The factorization follows Algorithm 1 of Salvaña (2026). It never stores the covariance matrix, and it stores each block of the Cholesky factor in the lowest of these precisions that an error budget allows.

Partition $\bar{\boldsymbol{\Sigma}}$ into an $n_B \times n_B$ grid of blocks of $n_b = 120$ consecutive points, so that $n_B = 12$. Let $m$ be a known lower bound on the smallest eigenvalue, $0 < m \le \lambda_{\min}(\bar{\boldsymbol{\Sigma}})$. Here $\bar{\boldsymbol{\Sigma}}$ is a nonnegative definite matrix plus a diagonal matrix that holds the relative nugget effects, so $m = \min_j \eta_j$ is available at no cost. For an error-budget parameter $0 < \tau < 1$ the budget of each block is
$$
b = \frac{\tau\, m}{n_B}.
$$

Let $\widetilde{\mathbf{L}}$ denote the stored factor. At block column $k$ the diagonal block is computed and kept in `fp64`,
$$
\widetilde{\mathbf{L}}_{kk} = \operatorname{chol}\Big( \bar{\boldsymbol{\Sigma}}_{kk} - \sum_{j < k} \widetilde{\mathbf{L}}_{kj} \widetilde{\mathbf{L}}_{kj}^\top \Big),
$$
and for each $i > k$ the off-diagonal block is computed in `fp64` as
$$
\mathbf{A}_{ik} = \Big( \bar{\boldsymbol{\Sigma}}_{ik} - \sum_{j < k} \widetilde{\mathbf{L}}_{ij} \widetilde{\mathbf{L}}_{kj}^\top \Big) \widetilde{\mathbf{L}}_{kk}^{-\top}.
$$
The block $\mathbf{A}_{ik}$ is then converted to each candidate format $p$ in the order `fp8`, `bf16`, `fp32`, read back in `fp64`, and stored in the first format with finite entries that satisfies
$$
\big\| \mathrm{fl}_p(\mathbf{A}_{ik}) - \mathbf{A}_{ik} \big\|_F\, \big\| \widetilde{\mathbf{L}}_{kk} \big\|_2 \le b,
$$
where $\mathrm{fl}_p$ denotes conversion to format $p$, $\|\cdot\|_F$ is the Frobenius norm and $\|\cdot\|_2$ the spectral norm. If no candidate qualifies, the block stays in `fp64`. The left-hand side bounds the change that storing the block in format $p$ causes in the covariance matrix implied by the stored factor, $\widetilde{\boldsymbol{\Sigma}} = \widetilde{\mathbf{L}} \widetilde{\mathbf{L}}^\top$.

Under this rule, and treating `fp64` arithmetic as exact, the Cholesky factorization of every diagonal block exists, so the factorization cannot break down, and the completed factor satisfies
$$
\big\| \widetilde{\boldsymbol{\Sigma}} - \bar{\boldsymbol{\Sigma}} \big\|_F \le \tau\, m \le \tau\, \lambda_{\min}(\bar{\boldsymbol{\Sigma}}),
$$
which implies that $\widetilde{\boldsymbol{\Sigma}}$ is positive definite. All arithmetic is done in `fp64`. Covariance blocks are formed when they are needed and discarded after use, and the log determinant is accumulated from the diagonal blocks, $\log \det \widetilde{\boldsymbol{\Sigma}} = 2 \sum_k \sum_r \log (\widetilde{\mathbf{L}}_{kk})_{rr}$. The notebook uses $\tau = 0.99$.

In [ ]:
TAU = 0.99
CANDIDATES = {"fp8": torch.float8_e4m3fn, "bf16": torch.bfloat16, "fp32": torch.float32}      # lowest precision first
LARGEST = {"fp8": 448.0, "bf16": 3.39e38, "fp32": 3.40e38}                                    # largest finite number of each format
FORMAT_NAMES = ["fp8", "bf16", "fp32", "fp64"]

def choose_format(A, budget):
    # the lowest precision whose measured storage error fits the budget, fp64 if none does
    largest = float(A.abs().max())
    for name, dtype in CANDIDATES.items():
        if largest <= LARGEST[name] and float(torch.linalg.norm(A.to(dtype).to(torch.float64) - A)) <= budget:
            return name, dtype
    return "fp64", torch.float64

def mixed_precision_cholesky(model):
    blocks, nB = model.blocks, len(model.blocks)
    b = TAU * model.m / nB                                                         # budget of each block
    L, logdet = {}, 0.0
    count = {"fp8": 0, "bf16": 0, "fp32": 0, "fp64": 0}
    precision = np.full((nB, nB), np.nan)                                          # format of every factor block, for the figure
    for k in range(nB):
        # diagonal block, kept in fp64
        S = model.block(blocks[k], blocks[k])
        S.diagonal().add_(model.nugget[blocks[k]])
        row = [L[k, j].to(torch.float64) for j in range(k)]
        for Lkj in row:
            S.addmm_(Lkj, Lkj.T, alpha=-1.0)
        Lkk = torch.linalg.cholesky(S)
        L[k, k] = Lkk
        precision[k, k] = FORMAT_NAMES.index("fp64")
        logdet += 2.0 * float(torch.log(torch.diagonal(Lkk)).sum())
        norm_kk = float(torch.linalg.matrix_norm(Lkk, ord=2))
        # off-diagonal blocks of column k, each stored in the lowest precision that fits the budget
        for i in range(k + 1, nB):
            A = model.block(blocks[i], blocks[k])
            for j in range(k):
                A.addmm_(L[i, j].to(torch.float64), row[j].T, alpha=-1.0)
            A = torch.linalg.solve_triangular(Lkk.T, A, upper=True, left=False)
            name, dtype = choose_format(A, b / norm_kk)
            L[i, k] = A.to(dtype)
            count[name] += 1
            precision[i, k] = FORMAT_NAMES.index(name)
    return L, logdet, count, precision

def solve(L, blocks, rhs):
    # solve (L L^T) x = rhs by block forward and back substitution
    x, nB = rhs.clone().reshape(-1, 1), len(blocks)
    for k in range(nB):
        s = x[blocks[k]]
        for j in range(k):
            s -= L[k, j].to(torch.float64) @ x[blocks[j]]
        x[blocks[k]] = torch.linalg.solve_triangular(L[k, k], s, upper=False)
    for k in range(nB - 1, -1, -1):
        s = x[blocks[k]]
        for i in range(k + 1, nB):
            s -= L[i, k].to(torch.float64).T @ x[blocks[i]]
        x[blocks[k]] = torch.linalg.solve_triangular(L[k, k].T, s, upper=True)
    return x.reshape(rhs.shape)

def negative_log_likelihood(theta, model):
    model.set_parameters(theta)
    L, logdet, count, precision = mixed_precision_cholesky(model)
    alpha = solve(L, model.blocks, model.y)
    sigma2 = float(model.y @ alpha) / model.N                                      # profiled variance of the hip
    stored_gb = sum(block.nbytes for block in L.values()) / 1e9
    N = model.N
    return 0.5 * (N * np.log(sigma2) + logdet + N + N * np.log(2 * np.pi)), sigma2, count, stored_gb, precision

### Size of the covariance matrix

The cell below builds the model on all training strides and prints the size of the covariance matrix that every evaluation of the likelihood has to factor.

In [ ]:
model = TrivariateMatern(F_train, Z_train)
N, nB = model.N, len(model.blocks)
print(f"covariance matrix          {N:,} x {N:,}   ({len(JOINTS)} joints x {model.n} strides)")
print(f"number of entries          {N * N:,}")
print(f"matrix in fp64             {N * N * 8 / 1e6:.1f} MB")
print(f"Cholesky factor in fp64    {N * (N + 1) / 2 * 8 / 1e6:.1f} MB")
print(f"Cholesky factor in fp32    {N * (N + 1) / 2 * 4 / 1e6:.1f} MB")
print(f"blocks                     {nB} x {nB}, each with up to {model.blocks[0].stop:,} points")
print(f"GPU                        {GPU_NAME}, {GPU_GB:.1f} GB")

## Maximum likelihood estimation

The ten parameters are estimated with the L-BFGS-B optimizer, with gradients from forward differences of the likelihood. The optimizer starts from the input ranges of the three univariate emulators of the first notebook, unit variances of the standardized outputs, nugget effects of one percent and moderate cross-correlations between the joints. The parameters are bounded to a plausible region, input ranges between 0.05 and 5 standardized units and relative nugget effects between $10^{-5}$ and $0.5$.

Every evaluation of the likelihood is one mixed-precision factorization of the $1{,}440 \times 1{,}440$ matrix. Each evaluation prints the negative log likelihood, the formats of the off-diagonal factor blocks and the current values of all the parameters. The ranges, variances and nugget effects are listed for the hip, the knee and the ankle in that order, and each cross-correlation is labeled with its pair of joints. The best parameters are saved to a checkpoint file after every improvement, and the cell resumes from that file when it exists. The cells after this one use the best parameters found so far, so they can also be run after the fit is interrupted.

In [ ]:
# starting values (hip, knee, ankle): the univariate input ranges, unit variances of the standardized outputs, 1 % nugget effects
l0 = np.array([0.354, 0.287, 0.313])                       # input ranges [standardized units]
var0 = np.array([1.0, 1.0, 1.0])                           # variances, standardized
nugget0 = np.array([0.01, 0.01, 0.01])                     # nugget effects, standardized
theta0 = np.concatenate([np.log(l0), [0.3, 0.3, 0.3], np.log(var0[1:] / var0[0]), np.log(nugget0 / var0[0])])
bounds = [(np.log(0.05), np.log(5.0))] * 3 + [(-2.0, 2.0)] * 3 + [(-3.0, 3.0)] * 2 + [(np.log(1e-5), np.log(0.5))] * 3

CHECKPOINT = f"trivariate_matern_checkpoint_{TARGET}.json"
MAX_ITERATIONS = 60
if os.path.exists(CHECKPOINT):
    theta0 = np.array(json.load(open(CHECKPOINT))["theta"])
    print("resuming from", CHECKPOINT)

best = {"nll": np.inf, "theta": None, "evaluations": 0}
t_start = time.time()
torch.cuda.reset_peak_memory_stats()

def objective(theta):
    # a trial point far from the best point so far is rejected without a factorization
    if best["theta"] is not None and np.max(np.abs(theta - best["theta"])) > 0.7:
        return best["nll"] + 1e6 * float(np.max(np.abs(theta - best["theta"])))
    t0 = time.time()
    nll, sigma2, count, stored_gb, precision = negative_log_likelihood(theta, model)
    best["evaluations"] += 1
    if nll < best["nll"]:
        best.update(nll=nll, theta=theta.copy(), sigma2=sigma2, count=count, stored_gb=stored_gb, precision=precision)
        json.dump({"theta": theta.tolist(), "nll": nll}, open(CHECKPOINT, "w"))
    p = model.parameters
    print(f"evaluation {best['evaluations']:4d}  negative log likelihood {nll:13.2f}  blocks fp8/bf16/fp32/fp64 {'/'.join(str(c) for c in count.values())}  [{time.time() - t0:.1f} s]\n"
          f"      input ranges {np.round(p['l'], 3).tolist()}  "
          f"variances [BW²] {np.round(sigma2 * p['v'] * scale**2, 4).tolist()}  nugget effects [BW²] {[f'{x:.2e}' for x in sigma2 * p['eta'] * scale**2]}  "
          f"cross-correlations hip-knee {p['rho'][0, 1]:.3f}, hip-ankle {p['rho'][0, 2]:.3f}, knee-ankle {p['rho'][1, 2]:.3f}", flush=True)
    return nll

result = minimize(objective, theta0, method="L-BFGS-B", bounds=bounds, options={"maxiter": MAX_ITERATIONS, "eps": 1e-3})
print(f"{result.message}, {result.nit} iterations, {best['evaluations']} likelihood evaluations, {time.time() - t_start:.0f} s")

The estimates are reported per joint in the units of the first notebook, followed by the cross-correlations $\rho_{jk}$ between the joints at the same stride and by a summary of the factorization at the estimates.

In [ ]:
model.set_parameters(best["theta"])
p, sigma2 = model.parameters, best["sigma2"]
estimates = pd.DataFrame({"input range [standardized units]": p["l"],
                          "variance [BW²]": sigma2 * p["v"] * scale**2, "nugget effect [BW²]": sigma2 * p["eta"] * scale**2}, index=JOINTS).T
estimates.columns.name = "joint"
shown = estimates.apply(lambda col: col.map("{:.3f}".format))
shown.loc["nugget effect [BW²]"] = estimates.loc["nugget effect [BW²]"].map("{:.2e}".format)
print(f"negative log likelihood {best['nll']:,.1f}")
shown

In [ ]:
pd.DataFrame(p["rho"], index=JOINTS, columns=JOINTS).round(3)

In [ ]:
N, nB = model.N, len(model.blocks)
pd.Series({"N": f"{N:,}", "blocks": f"{nB} x {nB}",
           "off-diagonal blocks in fp8 / bf16 / fp32 / fp64": " / ".join(str(c) for c in best["count"].values()),
           "stored Cholesky factor [MB]": f"{best['stored_gb'] * 1e3:.1f}",
           "Cholesky factor in fp64 [MB]": f"{N * (N + 1) / 2 * 8 / 1e6:.1f}",
           "covariance matrix in fp64 [MB]": f"{N * N * 8 / 1e6:.1f}",
           "peak GPU memory [MB]": f"{torch.cuda.max_memory_allocated() / 1e6:.1f}"}, name="mixed-precision factorization").to_frame()

### Precision assigned to each block

The figure shows the storage precision that the rule assigned to every block of the Cholesky factor at the estimates, in the layout of the per-block precision figure of Salvaña (2026). The diagonal blocks are always kept in `fp64`, and the percentages in the title refer to the off-diagonal blocks.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

PRECISION_COLORS = {"fp8": "#5E3C99", "bf16": "#1FBEC0", "fp32": "#F26F22", "fp64": "#4D4D4D"}     # purple / teal / orange / dark gray

def plot_precision(precision, count, N, block_size=120):
    nB, total = precision.shape[0], sum(count.values())
    cmap = ListedColormap([PRECISION_COLORS[name] for name in FORMAT_NAMES])
    cmap.set_bad("white")
    fig, ax = plt.subplots(figsize=(6.4, 6.6), constrained_layout=True)
    ax.imshow(np.ma.masked_invalid(precision), cmap=cmap, norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N), origin="upper", interpolation="nearest")
    ax.set_title("   ".join(f"{name} {100 * count[name] / total:.1f}%" for name in FORMAT_NAMES), pad=12)
    ax.set_xlabel("block column $k$"); ax.set_ylabel("block row $i$")
    fig.legend(handles=[Patch(facecolor=PRECISION_COLORS[name], label=name) for name in FORMAT_NAMES], loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(f"Per-block precision of the factor\n(block size $n_b$ = {block_size}, {nB} $\\times$ {nB} blocks, $N$ = {N:,})", fontsize=12)
    fig.savefig(f"trivariate_precision_per_block_{TARGET}.png", dpi=200, bbox_inches="tight")
    plt.show()

plot_precision(best["precision"], best["count"], model.N)

## References

Apanasovich, T. V., Genton, M. G. and Sun, Y. (2012). A valid Matérn class of cross-covariance functions for multivariate random fields with any number of components. *Journal of the American Statistical Association*, 107, 180-193.

Gneiting, T., Kleiber, W. and Schlather, M. (2010). Matérn cross-covariance functions for multivariate random fields. *Journal of the American Statistical Association*, 105, 1167-1177.

Salvaña, M. L. O. (2026). Adaptive block mixed-precision Cholesky factorization for large-scale space-time Gaussian processes. In *Proceedings of the SC26 Workshops, HPC for Environmental and Earth Sciences (HPC4EES)*, Chicago, IL, USA, November 15-20, 2026.